# Weather forecast

Gets the 5 day / 3 hour forecast from OpenWeather for every city in the `city` table and appends it to `weather_forecast`. The API key is read from `.env`.

In [1]:
import pandas as pd
from getpass import getpass
import requests
from datetime import datetime
from pytz import timezone
import os
from dotenv import load_dotenv

### Setup

In [2]:
def load_api_key():
    load_dotenv()  # reads .env into environment variables
    api_key = os.getenv("api_key")
    if api_key:
        return api_key
        print("api_key is set")
    else:
        print('api_key is not set')
        

In [3]:
def configure_pandas_display():
    """Show full DataFrames without truncation."""

    pd.set_option('display.max_rows', None)      # show all rows
    pd.set_option('display.max_columns', None)   # show all columns
    pd.set_option('display.max_colwidth', None)  # don't cut off long cell text
    pd.set_option('display.width', None)         # use the full terminal/notebook width

In [4]:
def build_connection_string(schema="gans", host="127.0.0.1", user="root", port=3306):
    """Ask for the MySQL password and return a SQLAlchemy connection string."""

    # the name of the MySQL database to connect to
    # the server address — 127.0.0.1 means "this same computer" (localhost)
    # the MySQL username to log in with
    # the network port MySQL listens on (3306 is MySQL's default)

    password = getpass('Enter MySQL password: ')      # the password for that MySQL user

    # Build the full connection URL that SQLAlchemy/pymysql needs to connect:
    # format: mysql+pymysql://<user>:<password>@<host>:<port>/<schema>
    connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'
    return connection_string

### Getting the forecast
One API call per city. `forecast_time` comes back in UTC, `data_retrieved_at` is Berlin time.

In [5]:
def fetch_weather_forecast(city_id, lat, lon, api_key):
    """Get the 5-day / 3-hour forecast for one city as a list of dicts."""
    berlin_timezone = timezone('Europe/Berlin')
    response = requests.get(
        "https://api.openweathermap.org/data/2.5/forecast",
        params={"lat": lat, "lon": lon, "appid": api_key, "units": "metric"},
    )
    response.raise_for_status()   # stop with a clear error if the request fails
    forecast_json = response.json()
    retrieval_time = datetime.now(berlin_timezone).strftime("%Y-%m-%d %H:%M:%S")

    rows = []
    for item in forecast_json["list"]:
        rows.append({
            "city_id": city_id,
            "forecast_time": item.get("dt_txt"),
            "temperature": item["main"].get("temp"),
            "forecast": item["weather"][0].get("main"),
            "rain_in_last_3h": item.get("rain", {}).get("3h", 0),
            "wind_speed": item["wind"].get("speed"),
            "data_retrieved_at": retrieval_time,
        })
    return rows

In [6]:
def get_cities(connection_string):
    """Load city ids and coordinates from the database."""
    return pd.read_sql(
        'SELECT city_id, name, latitude, longitude FROM city',
        con=connection_string,
    )

In [7]:
def collect_forecasts(city_df, api_key):
    """Fetch forecasts for every city and return one DataFrame."""
    records = []
    for _, city in city_df.iterrows():
        records.extend(fetch_weather_forecast(
            city['city_id'], city['latitude'], city['longitude'], api_key
        ))

    forecast_df = pd.DataFrame(records)
    forecast_df["forecast_time"] = pd.to_datetime(forecast_df["forecast_time"])
    forecast_df["data_retrieved_at"] = pd.to_datetime(forecast_df["data_retrieved_at"])
    return forecast_df

### Save to MySQL

In [8]:
def save_forecasts(forecast_df, connection_string):
    """Append forecasts to the weather_forecast table."""
    return forecast_df.to_sql(
        name='weather_forecast', con=connection_string,
        if_exists='append', index=False,
    )

### Run

In [9]:
def run_weather_pipeline():
    """Run the whole flow: load key, read cities, fetch, save."""
    configure_pandas_display()
    api_key = load_api_key()
    connection_string = build_connection_string()
    city_df = get_cities(connection_string)
    forecast_df = collect_forecasts(city_df, api_key)
    rows_saved = save_forecasts(forecast_df, connection_string)
    print(f"Saved {rows_saved} rows")
    return forecast_df

In [10]:
forecast_df = run_weather_pipeline()
forecast_df.head()

Saved 1200 rows


,city_id,forecast_time,temperature,forecast,rain_in_last_3h,wind_speed,data_retrieved_at
0,1,2026-10-14 18:00:00,14.93,Clouds,0.0,2.20,2026-10-14 17:01:24
1,1,2026-10-14 21:00:00,12.48,Clouds,0.0,1.66,2026-10-14 17:01:24
2,1,2026-10-15 00:00:00,9.49,Clouds,0.0,1.22,2026-10-14 17:01:24
3,1,2026-10-15 03:00:00,7.86,Clouds,0.0,1.35,2026-10-14 17:01:24
4,1,2026-10-15 06:00:00,7.06,Clouds,0.0,1.60,2026-10-14 17:01:24
